Печатную и электронную версии *Think Python 3e* можно заказать на
[Bookshop.org](https://bookshop.org/a/98697/9781098155438) и
[Amazon](https://www.amazon.com/_/dp/1098155432?smid=ATVPDKIKX0DER&_encoding=UTF8&tag=oreilly20-20&_encoding=UTF8&tag=greenteapre01-20&linkCode=ur2&linkId=e2a529f94920295d27ec8a06e757dc7c&camp=1789&creative=9325).

In [1]:
from os.path import basename, exists

def download(url):
    filename = basename(url)
    if not exists(filename):
        from urllib.request import urlretrieve

        local, _ = urlretrieve(url, filename)
        print("Downloaded " + str(local))
    return filename

download('https://github.com/AllenDowney/ThinkPython/raw/v3/thinkpython.py');
download('https://github.com/AllenDowney/ThinkPython/raw/v3/diagram.py');
download('https://github.com/ramalho/jupyturtle/releases/download/2024-03/jupyturtle.py');

import thinkpython

# Классы и объекты

К этому моменту мы уже определили классы и создали объекты, которые представляют время суток и день в году.
Мы написали методы, которые создают, изменяют и выполняют вычисления с этими объектами.

В этой главе мы продолжим знакомство с объектно-ориентированным программированием (ООП), определив классы, представляющие геометрические объекты: точки, линии, прямоугольники и окружности.
Мы напишем методы для создания и изменения этих объектов и будем использовать модуль `jupyturtle` для их рисования.

Эти классы я буду использовать, чтобы продемонстрировать темы ООП, включая идентичность и эквивалентность объектов, поверхностное и глубинное копирование и полиморфизм.

## Создание точки

В компьютерной графике положение на экране часто представляется парой координат в плоскости `x`–`y`.
По договорённости точка `(0, 0)` обычно соответствует левому верхнему углу экрана, а `(x, y)` обозначает точку на `x` единиц правее и `y` единиц ниже начала координат.
По сравнению с привычной из математики декартовой системой координат ось `y` здесь направлена вниз.

В Python точку можно представить несколькими способами:

-   хранить координаты отдельно в двух переменных `x` и `y`;

-   хранить координаты элементами списка или кортежа;

-   создать новый тип, представляющий точки как объекты.

В духе объектно-ориентированного программирования лучше всего создать новый тип.
Начнём с определения класса `Point`.

In [2]:
class Point:
    """Represents a point in 2-D space."""
    
    def __init__(self, x, y):
        self.x = x
        self.y = y
        
    def __str__(self):
        return f'Point({self.x}, {self.y})'

Метод `__init__` принимает координаты в качестве параметров и сохраняет их в атрибутах `x` и `y`.
Метод `__str__` возвращает строковое представление `Point`.

Теперь мы можем создать и вывести объект `Point` так.

In [3]:
start = Point(0, 0)
print(start)

Следующая диаграмма показывает состояние нового объекта.

In [4]:
from diagram import make_frame, make_binding

d1 = vars(start)
frame = make_frame(d1, name='Point', dy=-0.25, offsetx=0.18)
binding = make_binding('start', frame)

In [5]:
from diagram import diagram, adjust

width, height, x, y = [1.41, 0.89, 0.26, 0.5]
ax = diagram(width, height)
bbox = binding.draw(ax, x, y)
#adjust(x, y, bbox)

Как обычно, тип, определённый программистом, изображается прямоугольником: снаружи — имя типа, внутри — его атрибуты.

В целом пользовательские типы изменяемы, поэтому мы можем написать метод `translate`, который принимает два числа `dx` и `dy` и прибавляет их к атрибутам `x` и `y`.

In [6]:
%%add_method_to Point

    def translate(self, dx, dy):
        self.x += dx
        self.y += dy

Эта функция переносит `Point` из одной точки плоскости в другую.
Если мы не хотим изменять существующий `Point`, можно воспользоваться `copy`, чтобы скопировать исходный объект и затем изменить копию.

In [7]:
from copy import copy

end1 = copy(start)
end1.translate(300, 0)
print(end1)

Эти действия можно собрать в другом методе — `translated`.

In [8]:
%%add_method_to Point

    def translated(self, dx=0, dy=0):
        point = copy(self)
        point.translate(dx, dy)
        return point

Подобно тому как встроенная функция `sort` изменяет список на месте, а `sorted` создаёт новый список, у нас есть метод `translate`, который изменяет `Point`, и `translated`, который создаёт новый.

Вот пример:

In [9]:
end2 = start.translated(0, 150)
print(end2)

В следующем разделе мы воспользуемся этими точками, чтобы определить и нарисовать линию.

## Создание линии

Теперь определим класс, представляющий отрезок между двумя точками.
Как обычно, начнём с методов `__init__` и `__str__`.

In [10]:
class Line:
    def __init__(self, p1, p2):
        self.p1 = p1
        self.p2 = p2
        
    def __str__(self):
        return f'Line({self.p1}, {self.p2})'

С этими двумя методами мы можем создать и вывести объект `Line`, который будет представлять ось `x`.

In [11]:
line1 = Line(start, end1)
print(line1)

Когда мы вызываем `print` и передаём ему `line`, функция вызывает у него `__str__`.
Метод `__str__` использует f-строку, чтобы создать строковое представление линии.

В f-строке две выражения в фигурных скобках: `self.p1` и `self.p2`.
При их вычислении получаются объекты `Point`.
Затем при преобразовании их в строку вызывается метод `__str__` из класса `Point`.

Поэтому, выводя `Line`, мы видим строковые представления объектов `Point`.

Следующая диаграмма показывает состояние этого объекта `Line`.

In [12]:
from diagram import Binding, Value, Frame

d1 = vars(line1.p1)
frame1 = make_frame(d1, name='Point', dy=-0.25, offsetx=0.17)

d2 = vars(line1.p2)
frame2 = make_frame(d2, name='Point', dy=-0.25, offsetx=0.17)

binding1 = Binding(Value('start'), frame1, dx=0.4)
binding2 = Binding(Value('end'), frame2, dx=0.4)
frame3 = Frame([binding1, binding2], name='Line', dy=-0.9, offsetx=0.4, offsety=-0.25)

binding = make_binding('line1', frame3)

In [13]:
width, height, x, y = [2.45, 2.12, 0.27, 1.76]
ax = diagram(width, height)
bbox = binding.draw(ax, x, y)
#adjust(x, y, bbox)

Строковые представления и объектные диаграммы полезны для отладки, но цель этого примера — вывести графику, а не текст!
Поэтому мы воспользуемся модулем `jupyturtle`, чтобы рисовать линии на экране.

Как и в [главе 4](section_turtle_module), мы вызовем `make_turtle`, чтобы создать объект `Turtle` и небольшое окно для рисования.
Для рисования линий используем две новые функции из модуля `jupyturtle`:

* `jumpto` — принимает две координаты и перемещает `Turtle` в указанное место без рисования линии, и

* `moveto` — перемещает `Turtle` из текущего положения в указанное место, рисуя линию между ними.

Вот как их импортировать.

In [14]:
from jupyturtle import make_turtle, jumpto, moveto

А вот метод, который рисует `Line`.

In [15]:
%%add_method_to Line

    def draw(self):
        jumpto(self.p1.x, self.p1.y)
        moveto(self.p2.x, self.p2.y)

Чтобы показать, как это работает, я создам вторую линию, представляющую ось `y`.

In [16]:
line2 = Line(start, end2)
print(line2)

А потом нарисую оси.

In [17]:
make_turtle()
line1.draw()
line2.draw()

По мере того как мы будем определять и рисовать новые объекты, эти линии нам ещё пригодятся.
Но прежде поговорим об эквивалентности и идентичности объектов.

## Эквивалентность и идентичность

Предположим, мы создадим две точки с одинаковыми координатами.

In [18]:
p1 = Point(200, 100)
p2 = Point(200, 100)

Если применить к ним оператор `==`, получим поведение по умолчанию для пользовательских типов — результат `True` только если это один и тот же объект, что здесь не так.

In [19]:
p1 == p2

Чтобы изменить это поведение, можно определить специальный метод `__eq__`, задающий, что значит равенство двух объектов `Point`.

In [20]:
%%add_method_to Point

def __eq__(self, other):
    return (self.x == other.x) and (self.y == other.y)

В этом определении две точки считаются равными, если равны их атрибуты.
Теперь при использовании оператора `==` вызывается метод `__eq__`, который показывает, что `p1` и `p2` считаются равными.

In [21]:
p1 == p2

Но оператор `is` по-прежнему показывает, что это разные объекты.

In [22]:
p1 is p2

Переопределить оператор `is` невозможно — он всегда проверяет идентичность объектов.
А вот оператор `==` в пользовательских типах можно переопределить, чтобы он проверял эквивалентность.
Причём вы сами решаете, что считать эквивалентным.

## Создание прямоугольника

Теперь определим класс, который представляет и рисует прямоугольники.
Чтобы не усложнять, будем считать, что прямоугольники расположены вертикально или горизонтально, без наклона.
Какие атрибуты нам нужны, чтобы задать положение и размер прямоугольника?

Возможны по крайней мере два варианта:

-   указать ширину и высоту прямоугольника и координаты одного из углов.

-   указать координаты двух противоположных углов.

Пока трудно сказать, какой вариант лучше, поэтому реализуем первый.
Вот определение класса.

In [23]:
class Rectangle:
    """Represents a rectangle. 

    attributes: width, height, corner.
    """
    def __init__(self, width, height, corner):
        self.width = width
        self.height = height
        self.corner = corner
        
    def __str__(self):
        return f'Rectangle({self.width}, {self.height}, {self.corner})'

Как обычно, `__init__` сохраняет параметры в атрибутах, а `__str__` возвращает строковое представление объекта.
Теперь мы можем создать объект `Rectangle`, используя `Point` в качестве координат верхнего левого угла.

In [24]:
corner = Point(30, 20)
box1 = Rectangle(100, 50, corner)
print(box1)

Следующая диаграмма показывает состояние этого объекта.

In [25]:
from diagram import Binding, Value

def make_rectangle_binding(name, box, **options):
    d1 = vars(box.corner)
    frame_corner = make_frame(d1, name='Point', dy=-0.25, offsetx=0.07)

    d2 = dict(width=box.width, height=box.height)
    frame = make_frame(d2, name='Rectangle', dy=-0.25, offsetx=0.45)
    binding = Binding(Value('corner'), frame1, dx=0.92, draw_value=False, **options)
    frame.bindings.append(binding)

    binding = Binding(Value(name), frame)
    return binding, frame_corner

binding_box1, frame_corner1 = make_rectangle_binding('box1', box1)

In [26]:
from diagram import Bbox

width, height, x, y = [2.83, 1.49, 0.27, 1.1]
ax = diagram(width, height)
bbox1 = binding_box1.draw(ax, x, y)
bbox2 = frame_corner1.draw(ax, x+1.85, y-0.6)
bbox = Bbox.union([bbox1, bbox2])
#adjust(x, y, bbox)

Чтобы нарисовать прямоугольник, используем следующий метод: он создаёт четыре объекта `Point`, представляющие углы.

In [27]:
%%add_method_to Rectangle

    def make_points(self):
        p1 = self.corner
        p2 = p1.translated(self.width, 0)
        p3 = p2.translated(0, self.height)
        p4 = p3.translated(-self.width, 0)
        return p1, p2, p3, p4

Затем создадим четыре объекта `Line`, соответствующие сторонам.

In [28]:
%%add_method_to Rectangle

    def make_lines(self):
        p1, p2, p3, p4 = self.make_points()
        return Line(p1, p2), Line(p2, p3), Line(p3, p4), Line(p4, p1)

После этого нарисуем стороны.

In [29]:
%%add_method_to Rectangle

    def draw(self):
        lines = self.make_lines()
        for line in lines:
            line.draw()

Вот пример.

In [30]:
make_turtle()
line1.draw()
line2.draw()
box1.draw()

На рисунке есть две линии, обозначающие оси.

## Изменение прямоугольников

Рассмотрим два метода, изменяющих прямоугольники: `grow` и `translate`.
Мы увидим, что `grow` работает как ожидалось, а в `translate` скрыта тонкая ошибка.
Попробуйте обнаружить её до того, как я объясню.

Метод `grow` принимает два числа `dwidth` и `dheight` и прибавляет их к атрибутам `width` и `height` прямоугольника.

In [31]:
%%add_method_to Rectangle

    def grow(self, dwidth, dheight):
        self.width += dwidth
        self.height += dheight

Этот пример демонстрирует результат: мы сделаем копию `box1` и вызовем `grow` для копии.

In [32]:
box2 = copy(box1)
box2.grow(60, 40)
print(box2)

Если нарисовать `box1` и `box2`, можно убедиться, что `grow` сработал правильно.

In [33]:
make_turtle()
line1.draw()
line2.draw()
box1.draw()
box2.draw()

Теперь посмотрим на `translate`.
Этот метод принимает два числа `dx` и `dy` и перемещает прямоугольник на указанные расстояния по осям `x` и `y`.

In [34]:
%%add_method_to Rectangle

    def translate(self, dx, dy):
        self.corner.translate(dx, dy)

Чтобы показать результат, переместим `box2` вправо и вниз.

In [35]:
box2.translate(30, 20)
print(box2)

Теперь посмотрим, что произойдёт, если снова нарисовать `box1` и `box2`.

In [36]:
make_turtle()
line1.draw()
line2.draw()
box1.draw()
box2.draw()

Похоже, переместились оба прямоугольника, чего мы не хотели!
В следующем разделе объясняется, что пошло не так.

## Глубокое копирование

Когда мы используем `copy` для дублирования `box1`, копируется объект `Rectangle`, но не объект `Point`, содержащийся в нём.
Поэтому `box1` и `box2` — разные объекты, как и задумывалось.

In [37]:
box1 is box2

Но их атрибуты `corner` указывают на один и тот же объект.

In [38]:
box1.corner is box2.corner

На следующей диаграмме показано состояние этих объектов.

In [39]:
from diagram import Stack
from copy import deepcopy

binding_box1, frame_corner1 = make_rectangle_binding('box1', box1)
binding_box2, frame_corner2 = make_rectangle_binding('box2', box2, dy=0.4)
binding_box2.value.bindings.reverse()

stack = Stack([binding_box1, binding_box2], dy=-1.3)

In [40]:
from diagram import Bbox

width, height, x, y = [2.76, 2.54, 0.27, 2.16]
ax = diagram(width, height)
bbox1 = stack.draw(ax, x, y)
bbox2 = frame_corner1.draw(ax, x+1.85, y-0.6)
bbox = Bbox.union([bbox1, bbox2])
# adjust(x, y, bbox)

То, что делает `copy`, называется **поверхностным копированием**, потому что копируется сам объект, но не объекты внутри него.
Поэтому изменение `width` или `height` одного прямоугольника не затрагивает другой, а изменение атрибутов общего объекта `Point` отражается на обоих!
Такое поведение сбивает с толку и приводит к ошибкам.

К счастью, модуль `copy` предоставляет функцию `deepcopy`, которая копирует не только сам объект, но и объекты, на которые он ссылается, и те, на которые ссылаются они, и так далее.
Эта операция называется **глубоким копированием**.

Для демонстрации создадим новый `Rectangle`, содержащий новый `Point`.

In [41]:
corner = Point(20, 20)
box3 = Rectangle(100, 50, corner)
print(box3)

И сделаем его глубокую копию.

In [42]:
from copy import deepcopy

box4 = deepcopy(box3)

Мы можем убедиться, что два объекта `Rectangle` содержат разные объекты `Point`.

In [43]:
box3.corner is box4.corner

Поскольку `box3` и `box4` полностью независимы, мы можем изменять один, не затрагивая другой.
Для примера переместим `box3` и увеличим `box4`.

In [44]:
box3.translate(50, 30)
box4.grow(100, 60)

И мы можем убедиться, что всё работает как нужно.

In [45]:
make_turtle()
line1.draw()
line2.draw()
box3.draw()
box4.draw()

## Полиморфизм

В предыдущем примере мы вызывали метод `draw` у двух объектов `Line` и двух объектов `Rectangle`.
То же самое можно сделать короче, составив список объектов.

In [46]:
shapes = [line1, line2, box3, box4]

Элементы этого списка разных типов, но у всех есть метод `draw`, поэтому можно пройти по списку циклом и вызвать `draw` для каждого.

In [47]:
make_turtle()

for shape in shapes:
    shape.draw()

Первый и второй раз в цикле переменная `shape` ссылается на объект `Line`, поэтому вызывается метод, определённый в классе `Line`.

Третий и четвёртый раз `shape` ссылается на объект `Rectangle`, и вызывается метод из класса `Rectangle`.

В некотором смысле каждый объект знает, как нарисовать себя.
Эта возможность называется **полиморфизмом**.
Слово происходит от греческих корней, означающих «многообразие форм».
В ООП полиморфизм — это способность разных типов предоставлять одни и те же методы, что позволяет выполнять множество операций — например, рисование фигур — вызывая один и тот же метод у объектов разных типов.

В упражнении в конце главы вы определите новый класс, представляющий окружность и предоставляющий метод `draw`.
Тогда вы сможете использовать полиморфизм, чтобы рисовать линии, прямоугольники и окружности.

## Отладка

В этой главе мы столкнулись с тонкой ошибкой: мы создали `Point`, который использовали два объекта `Rectangle`, а затем изменили этот `Point`.
В общем случае есть два способа избежать подобных проблем: не делиться объектами и не изменять их.

Чтобы не делиться объектами, можно использовать глубокое копирование, как мы сделали в этой главе.

Чтобы не изменять объекты, рассмотрите возможность заменить нечистые функции типа `translate` чистыми функциями, например `translated`.
Вот вариант `translated`, который создаёт новый `Point` и никогда не меняет его атрибуты.

In [48]:
    def translated(self, dx=0, dy=0):
        x = self.x + dx
        y = self.y + dy
        return Point(x, y)

В Python есть возможности, позволяющие легче избегать изменения объектов.
Они выходят за рамки этой книги, но если вам интересно, спросите у виртуального помощника: «Как сделать объект Python неизменяемым?»

Создание нового объекта требует больше времени, чем изменение существующего, но на практике эта разница редко имеет значение.
Программы, избегающие общих объектов и нечистых функций, обычно легче разрабатывать, тестировать и отлаживать — а лучшая отладка та, которую не приходится делать.

## Глоссарий

**shallow copy:**  
операция копирования, которая не копирует вложенные объекты.

**deep copy:**  
операция копирования, которая также копирует вложенные объекты.

**polymorphism:**  
способность метода или оператора работать с объектами разных типов.

## Упражнения

In [ ]:
# This cell tells Jupyter to provide detailed debugging information
# when a runtime error occurs. Run it before working on the exercises.

%xmode Verbose

### Вопрос виртуальному помощнику

Во всех последующих упражнениях подумайте о том, чтобы обратиться за помощью к виртуальному ассистенту.
Если решите это сделать, включите в запрос определения классов `Point`, `Line` и `Rectangle` — иначе помощник лишь предположит их атрибуты и методы, и сгенерированный код не заработает.

### Упражнение

Напишите метод `__eq__` для класса `Line`, который возвращает `True`, если объекты `Line` ссылаются на эквивалентные объекты `Point`, независимо от порядка.

Для начала можно использовать следующую заготовку.

In [49]:
%%add_method_to Line

def __eq__(self, other):
    return None

In [50]:
# Solution goes here

Вы можете использовать эти примеры для проверки кода.

In [51]:
start1 = Point(0, 0)
start2 = Point(0, 0)
end = Point(200, 100)

Здесь результат должен быть `True`, потому что объекты `Line` ссылаются на эквивалентные объекты `Point` в том же порядке.

In [52]:
line_a = Line(start1, end)
line_b = Line(start2, end)
line_a == line_b    # should be True

Этот пример тоже должен быть `True`, потому что объекты `Line` ссылаются на эквивалентные объекты `Point` в обратном порядке.

In [53]:
line_c = Line(end, start1)
line_a == line_c     # should be True

Эквивалентность должна быть транзитивной: если `line_a` эквивалентна `line_b`, а `line_a` эквивалентна `line_c`, то `line_b` и `line_c` тоже должны быть эквивалентны.

In [54]:
line_b == line_c     # should be True

Здесь результат должен быть `False`, потому что объекты `Line` ссылаются на разные точки.

In [55]:
line_d = Line(start1, start2)
line_a == line_d    # should be False

### Упражнение

Напишите метод `midpoint` класса `Line`, который вычисляет середину отрезка и возвращает результат в виде объекта `Point`.

Для начала можно использовать следующую заготовку.

In [56]:
%%add_method_to Line

    def midpoint(self):
        return Point(0, 0)

In [57]:
# Solution goes here

Этими примерами можно воспользоваться для проверки кода и рисования результата.

In [58]:
start = Point(0, 0)
end1 = Point(300, 0)
end2 = Point(0, 150)
line1 = Line(start, end1)
line2 = Line(start, end2)

In [59]:
mid1 = line1.midpoint()
print(mid1)

In [60]:
mid2 = line2.midpoint()
print(mid2)

In [61]:
line3 = Line(mid1, mid2)

In [62]:
make_turtle()

for shape in [line1, line2, line3]:
    shape.draw()

### Упражнение

Напишите метод `midpoint` класса `Rectangle`, который находит точку в центре прямоугольника и возвращает её как объект `Point`.

Для начала можно использовать следующую заготовку.

In [63]:
%%add_method_to Rectangle

    def midpoint(self):
        return Point(0, 0)

In [64]:
# Solution goes here

Вы можете использовать следующий пример для проверки кода.

In [65]:
corner = Point(30, 20)
rectangle = Rectangle(100, 80, corner)

In [66]:
mid = rectangle.midpoint()
print(mid)

In [67]:
diagonal = Line(corner, mid)

In [68]:
make_turtle()

for shape in [line1, line2, rectangle, diagonal]:
    shape.draw()

### Упражнение

Напишите метод `make_cross` класса `Rectangle`, который:

1. с помощью `make_lines` получает список объектов `Line`, представляющих четыре стороны прямоугольника;

2. вычисляет середины этих четырёх линий;

3. создаёт и возвращает список из двух объектов `Line`, которые соединяют противоположные середины, образуя крест через центр прямоугольника.

Для начала можете использовать эту заготовку.

In [69]:
%%add_method_to Rectangle

    def make_diagonals(self):
        return []

In [70]:
# Solution goes here

Следующий пример можно использовать для проверки кода.

In [71]:
corner = Point(30, 20)
rectangle = Rectangle(100, 80, corner)

In [72]:
lines = rectangle.make_cross()

In [73]:
make_turtle()

rectangle.draw()
for line in lines:
    line.draw()

### Упражнение

Определите класс `Circle` с атрибутами `center` и `radius`, где `center` — объект `Point`, а `radius` — число.
Добавьте специальные методы `__init__` и `__str__`, а также метод `draw`, который использует функции `jupyturtle` для рисования окружности.

Вы можете использовать следующую функцию — это вариант функции `circle`, которую мы писали в Главе 4.

In [74]:
from jupyturtle import make_turtle, forward, left, right
import math
    
def draw_circle(radius):
    circumference = 2 * math.pi * radius
    n = 30
    length = circumference / n
    angle = 360 / n
    left(angle / 2)
    for i in range(n):
        forward(length)
        left(angle)

In [75]:
# Solution goes here

Для проверки кода можно воспользоваться следующим примером.
Начнём с квадратного прямоугольника шириной и высотой `100`.

In [76]:
corner = Point(20, 20)
rectangle = Rectangle(100, 100, corner)

Следующий код должен создать `Circle`, вписанный в квадрат.

In [77]:
center = rectangle.midpoint()
radius = rectangle.height / 2

circle = Circle(center, radius)
print(circle)

Если всё сработало правильно, следующий код нарисует окружность внутри квадрата (касаясь всех четырёх сторон).

In [78]:
make_turtle(delay=0.01)

rectangle.draw()
circle.draw()

[Think Python: 3rd Edition](https://allendowney.github.io/ThinkPython/index.html)

© 2024 [Allen B. Downey](https://allendowney.com)

Лицензия на код: [MIT License](https://mit-license.org/)

Лицензия на текст: [Creative Commons Attribution-NonCommercial-ShareAlike 4.0 International](https://creativecommons.org/licenses/by-nc-sa/4.0/)